In [2]:
%cd /content
!git clone https://github.com/AxelBadouel/Visual-Place-Recognition-Project.git
%cd /content/Visual-Place-Recognition-Project

/content
Cloning into 'Visual-Place-Recognition-Project'...
remote: Enumerating objects: 192, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 192 (delta 44), reused 36 (delta 22), pack-reused 112 (from 4)
Receiving objects: 100% (192/192), 1.41 MiB | 13.99 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/Visual-Place-Recognition-Project


In [14]:
import os
import glob
import subprocess
import pandas as pd

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
print("=== MegaLoc Test Folders ===")
for p in sorted(glob.glob("/content/drive/MyDrive/VPR_megaloc_logs/*")):
    if os.path.isdir(p):
        print(p)

print("\n=== CosPlace Test Folders ===")
for p in sorted(glob.glob("/content/drive/MyDrive/VPR_cosplace_logs/*")):
    if os.path.isdir(p):
        print(p)

=== MegaLoc Test Folders ===
/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_15-54-43
/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_16-47-53
/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-11-49
/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-45-58

=== CosPlace Test Folders ===
/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-14-23
/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-37-08
/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55
/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-47-17


In [7]:
def inspect_logs(base_dir):
    items = sorted(glob.glob(os.path.join(base_dir, "*")))
    # Keep only directories
    folders = [f for f in items if os.path.isdir(f)]

    for folder in folders:
        subdirs = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))]
        txt_count = len(glob.glob(os.path.join(folder, "preds", "*.txt")))

        # Benchmark identification
        benchmark = "Unknown"
        if txt_count == 1000:
            benchmark = "SF-XS"
        elif txt_count == 315:
            benchmark = "Tokyo-XS"
        elif txt_count == 823:
            benchmark = "SVOX Sun"
        elif txt_count == 854:
            benchmark = "SVOX Night"

        print(f"Folder: {os.path.basename(folder)} -> [{benchmark}]")
        print(f"  Path: {folder}")
        print(f"  Available Matchers: {[s for s in subdirs if s.startswith('preds_')]}")
        print(f"  Queries: {txt_count}")
        print("-" * 55)

print("=== MEGALOC LOGS ===")
inspect_logs("/content/drive/MyDrive/VPR_megaloc_logs")

print("\n=== COSPLACE LOGS ===")
inspect_logs("/content/drive/MyDrive/VPR_cosplace_logs")

=== MEGALOC LOGS ===
Folder: 2026-08-22_15-54-43 -> [SF-XS]
  Path: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_15-54-43
  Available Matchers: ['preds_superpoint-lg', 'preds_superglue', 'preds_loftr']
  Queries: 1000
-------------------------------------------------------
Folder: 2026-08-22_16-47-53 -> [Tokyo-XS]
  Path: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_16-47-53
  Available Matchers: ['preds_superpoint-lg', 'preds_superglue', 'preds_loftr']
  Queries: 315
-------------------------------------------------------
Folder: 2026-08-22_17-11-49 -> [SVOX Sun]
  Path: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-11-49
  Available Matchers: ['preds_superpoint-lg', 'preds_superglue', 'preds_loftr']
  Queries: 823
-------------------------------------------------------
Folder: 2026-08-22_17-45-58 -> [SVOX Night]
  Path: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-45-58
  Available Matchers: ['preds_superpoint-lg', 'preds_superglue', 'preds_loftr']
  Quer

In [8]:
log_bases = [
    "/content/drive/MyDrive/VPR_megaloc_logs",
    "/content/drive/MyDrive/VPR_cosplace_logs"
]

matchers_to_extract = ["superglue", "loftr"]

for base in log_bases:
    if not os.path.exists(base):
        continue
    folders = sorted([f for f in glob.glob(os.path.join(base, "*")) if os.path.isdir(f)])

    for folder in folders:
        preds_dir = os.path.join(folder, "preds")
        if not os.path.exists(preds_dir):
            continue

        for matcher in matchers_to_extract:
            inliers_dir = os.path.join(folder, f"preds_{matcher}")
            if not os.path.exists(inliers_dir):
                continue

            print(f"\n[RUNNING] {os.path.basename(base)}/{os.path.basename(folder)} with {matcher}...")

            cmd = [
                "python",
                "/content/Visual-Place-Recognition-Project/logistic_regression/dataset_constructor.py",
                "--preds_dir", preds_dir,
                "--inliers_dir", inliers_dir,
                "--num-preds", "20"
            ]

            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0:
                print(f"  ✓ Successfully created: {folder}/{matcher}.csv")
            else:
                print(f"  ✗ Error in {folder}/{matcher}:\n{result.stderr}")


[RUNNING] VPR_megaloc_logs/2026-08-22_15-54-43 with superglue...
  ✓ Successfully created: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_15-54-43/superglue.csv

[RUNNING] VPR_megaloc_logs/2026-08-22_15-54-43 with loftr...
  ✓ Successfully created: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_15-54-43/loftr.csv

[RUNNING] VPR_megaloc_logs/2026-08-22_16-47-53 with superglue...
  ✓ Successfully created: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_16-47-53/superglue.csv

[RUNNING] VPR_megaloc_logs/2026-08-22_16-47-53 with loftr...
  ✓ Successfully created: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_16-47-53/loftr.csv

[RUNNING] VPR_megaloc_logs/2026-08-22_17-11-49 with superglue...
  ✓ Successfully created: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-11-49/superglue.csv

[RUNNING] VPR_megaloc_logs/2026-08-22_17-11-49 with loftr...
  ✓ Successfully created: /content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-11-49/loftr.csv

[RUNNING] VPR_megaloc_logs/

In [9]:
all_csvs = sorted(glob.glob("/content/drive/MyDrive/VPR_*_logs/*/*.csv"))
print(f"Total CSVs generated: {len(all_csvs)}\n")

for csv_path in all_csvs:
    df = pd.read_csv(csv_path)
    parent_folder = os.path.basename(os.path.dirname(csv_path))
    model_type = "MegaLoc" if "megaloc" in csv_path else "CosPlace"
    print(f"[{model_type}] {parent_folder}/{os.path.basename(csv_path)}")
    print(f"   Rows: {len(df)} | Columns: {list(df.columns)} | Label counts: {dict(df['label'].value_counts())}")

Total CSVs generated: 16

[CosPlace] 2026-08-19_15-14-23/loftr.csv
   Rows: 1000 | Columns: ['inliers', 'label', 'query_file'] | Label counts: {1: np.int64(774), 0: np.int64(226)}
[CosPlace] 2026-08-19_15-14-23/superglue.csv
   Rows: 1000 | Columns: ['inliers', 'label', 'query_file'] | Label counts: {1: np.int64(768), 0: np.int64(232)}
[CosPlace] 2026-08-19_15-37-08/loftr.csv
   Rows: 315 | Columns: ['inliers', 'label', 'query_file'] | Label counts: {1: np.int64(267), 0: np.int64(48)}
[CosPlace] 2026-08-19_15-37-08/superglue.csv
   Rows: 315 | Columns: ['inliers', 'label', 'query_file'] | Label counts: {1: np.int64(256), 0: np.int64(59)}
[CosPlace] 2026-08-19_15-40-55/loftr.csv
   Rows: 823 | Columns: ['inliers', 'label', 'query_file'] | Label counts: {1: np.int64(505), 0: np.int64(318)}
[CosPlace] 2026-08-19_15-40-55/superglue.csv
   Rows: 823 | Columns: ['inliers', 'label', 'query_file'] | Label counts: {1: np.int64(489), 0: np.int64(334)}
[CosPlace] 2026-08-19_15-47-17/loftr.csv
   

In [12]:
!python /content/Visual-Place-Recognition-Project/logistic_regression/logistic_classifier_training.py \
    --train_inliers_df_csv "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55/superglue.csv" \
    --val_inliers_df_csv "/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-47-17/superglue.csv" \
    --out_dir "/content/drive/MyDrive/logistic_checkpoint.pth"

Threshold 0.2 | Acc: 0.8185 | % Re-ranked (Hard): 0.1%
Threshold 0.3 | Acc: 0.8888 | % Re-ranked (Hard): 9.0%
Threshold 0.4 | Acc: 0.9204 | % Re-ranked (Hard): 19.9%
Threshold 0.5 | Acc: 0.8888 | % Re-ranked (Hard): 27.0%
Threshold 0.6 | Acc: 0.8407 | % Re-ranked (Hard): 33.0%
Threshold 0.7 | Acc: 0.7904 | % Re-ranked (Hard): 38.3%
Threshold 0.8 | Acc: 0.6850 | % Re-ranked (Hard): 48.8%
Threshold 0.9 | Acc: 0.5562 | % Re-ranked (Hard): 61.9%

Selected Optimal Threshold: 0.5
Model saved to /content/drive/MyDrive/logistic_checkpoint.pth


In [13]:
checkpoint_path = "/content/drive/MyDrive/logistic_checkpoint.pth"

test_runs = [
    # MegaLoc Runs
    ("/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_15-54-43", "SF-XS", "MegaLoc"),
    ("/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_16-47-53", "Tokyo-XS", "MegaLoc"),
    ("/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-11-49", "SVOX Sun", "MegaLoc"),
    ("/content/drive/MyDrive/VPR_megaloc_logs/2026-08-22_17-45-58", "SVOX Night", "MegaLoc"),
    # CosPlace Runs
    ("/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-14-23", "SF-XS", "CosPlace"),
    ("/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-37-08", "Tokyo-XS", "CosPlace"),
    ("/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-40-55", "SVOX Sun", "CosPlace"),
    ("/content/drive/MyDrive/VPR_cosplace_logs/2026-08-19_15-47-17", "SVOX Night", "CosPlace"),
]

matchers = ["superglue", "loftr"]

print("=" * 105)
print(f"{'Model':<10} | {'Benchmark':<12} | {'Matcher':<10} | {'Total':<6} | {'Re-ranked (%)':<20} | {'Bypassed (%)':<20} | {'Adaptive R@1':<12}")
print("=" * 105)

for folder, benchmark, model in test_runs:
    for matcher in matchers:
        csv_file = os.path.join(folder, f"{matcher}.csv")
        inliers_dir = os.path.join(folder, f"preds_{matcher}")

        if not os.path.exists(csv_file) or not os.path.exists(inliers_dir):
            continue

        cmd = [
            "python",
            "/content/Visual-Place-Recognition-Project/logistic_regression/logistic_classifier_testing.py",
            "--test_inliers_df_csv", csv_file,
            "--inliers_dir", inliers_dir,
            "--classifier_weights", checkpoint_path,
            "--num-preds", "20"
        ]

        res = subprocess.run(cmd, capture_output=True, text=True)

        if res.returncode != 0:
            print(f"{model:<10} | {benchmark:<12} | {matcher:<10} | ERROR: {res.stderr.strip()[:60]}...")
            continue

        output = res.stdout
        total_q = [l.split(":")[-1].strip() for l in output.split("\n") if "Total Test Queries:" in l]
        reranked = [l.split(":")[-1].strip() for l in output.split("\n") if "Queries Re-ranked:" in l]
        recall = [l.split(":")[-1].strip() for l in output.split("\n") if "Final Recall@1 Accuracy:" in l or "Final Adaptive Recall@1:" in l]

        tot_str = total_q[0] if total_q else "N/A"
        rerank_str = reranked[0] if reranked else "N/A"
        rec_str = recall[0] if recall else "N/A"

        # Calculate bypassed string if rerank_str is available
        if "(" in rerank_str:
            pct_val = float(rerank_str.split("(")[-1].replace("%)", ""))
            bypass_str = f"{100.0 - pct_val:.2f}%"
        else:
            bypass_str = "N/A"

        print(f"{model:<10} | {benchmark:<12} | {matcher:<10} | {tot_str:<6} | {rerank_str:<20} | {bypass_str:<20} | {rec_str:<12}")

Model      | Benchmark    | Matcher    | Total  | Re-ranked (%)        | Bypassed (%)         | Adaptive R@1
MegaLoc    | SF-XS        | superglue  | 1000   | 476 (47.60%)         | 52.40%               | 0.8480      
MegaLoc    | SF-XS        | loftr      | 1000   | 48 (4.80%)           | 95.20%               | 0.8620      
MegaLoc    | Tokyo-XS     | superglue  | 315    | 159 (50.48%)         | 49.52%               | 0.9079      
MegaLoc    | Tokyo-XS     | loftr      | 315    | 23 (7.30%)           | 92.70%               | 0.9365      
MegaLoc    | SVOX Sun     | superglue  | 823    | 249 (30.26%)         | 69.74%               | 0.9222      
MegaLoc    | SVOX Sun     | loftr      | 823    | 71 (8.63%)           | 91.37%               | 0.9478      
MegaLoc    | SVOX Night   | superglue  | 854    | 178 (20.84%)         | 79.16%               | 0.9368      
MegaLoc    | SVOX Night   | loftr      | 854    | 27 (3.16%)           | 96.84%               | 0.9614      
CosPlace   | SF-XS 